# Lekcja 11 - Protokół Kontekstu Modelu (MCP)

**Protokół Kontekstu Modelu (MCP)** to otwarty standard, który umożliwia agentom dynamiczne odkrywanie i korzystanie z narzędzi, zasobów i źródeł danych w czasie działania. Zamiast na stałe wbudowywać narzędzia w agenta, MCP pozwala agentom łączyć się z zewnętrznymi serwerami, które udostępniają możliwości na żądanie.

W tej lekcji nauczysz się:
- Czym jest MCP i dlaczego jest ważny dla systemów agentowych
- Jak działa architektura klient-serwer MCP
- Jak budować agentów korzystających z odkrywania narzędzi w stylu MCP


## Konfiguracja

**Wymagania wstępne:**
- Projekt Microsoft Foundry z wdrożonym modelem
- Uruchom `az login` w celu uwierzytelnienia


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Czym jest Model Context Protocol (MCP)?

MCP definiuje standardowy sposób, w jaki agenci AI odkrywają i wchodzą w interakcje z zewnętrznymi narzędziami i źródłami danych:

- **Serwer MCP**: Udostępnia narzędzia, zasoby i podpowiedzi za pomocą standardowego protokołu
- **Klient MCP**: Środowisko uruchomieniowe agenta, które łączy się z serwerami i odkrywa dostępne funkcje
- **Dynamiczne odkrywanie**: Agenci nie potrzebują narzędzi wpisanych na sztywno — odkrywają, co jest dostępne podczas działania

To potężne narzędzie do budowy rozszerzalnych systemów agentów, w których nowe funkcje można dodawać bez modyfikowania kodu agenta.


## Jak działa MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (klient MCP) łączy się z serwerem MCP
2. Serwer odpowiada listą dostępnych narzędzi i ich schematami
3. Agent może następnie wywołać dowolne znalezione narzędzie podczas swojego rozumowania
4. Wyniki przepływają z powrotem tym samym protokołem


## Symulacja wykrywania narzędzia MCP

Ponieważ prawdziwy serwer MCP wymaga działającego procesu serwera, pokażemy wzorzec za pomocą funkcji `@tool`, które symulują to, co dostarczyłaby usługa zakwaterowania połączona z MCP.

W środowisku produkcyjnym narzędzia te byłyby wykrywane dynamicznie z serwera MCP, a nie definiowane lokalnie.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Tworzenie Agenta z narzędziami w stylu MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP w Produkcji

W środowisku produkcyjnym MCP umożliwia potężne wzorce:

- **Dynamiczne wykrywanie narzędzi**: Agenci łączą się z serwerami MCP i wykrywają narzędzia w czasie działania
- **Rozproszona architektura**: Dostawcy narzędzi mogą aktualizować niezależnie od agenta
- **Udostępnianie międzyorganizacyjne**: Zespoły mogą udostępniać możliwości poprzez serwery MCP, z których może korzystać każdy agent
- **Obsługa Microsoft Agent Framework**: MAF ma wbudowane wsparcie klienta MCP przez integrację `mcp`

Aby użyć prawdziwego serwera MCP z MAF, należy połączyć się przez `hosted_mcp_tool()` lub integrację klienta MCP.

**Dowiedz się więcej:**
- [Specyfikacja MCP](https://modelcontextprotocol.io/)
- [Wsparcie MCP Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Podsumowanie

W tej lekcji nauczyłeś się:
- **MCP** to otwarty standard do dynamicznego wykrywania narzędzi pomiędzy agentami a dostawcami narzędzi
- **Architektura klient-serwer** pozwala agentom odkrywać możliwości podczas działania
- MCP umożliwia **rozszerzalne, odseparowane systemy agentów**, gdzie narzędzia mogą być dodawane bez zmian w kodzie
- Microsoft Agent Framework zapewnia **wbudowane wsparcie MCP** do zastosowań produkcyjnych


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
